# Animal Detector Trainer
This notebook extracts frames from videos, prepares a YOLO dataset, trains/validates a detector, and computes center points from detections.

## Step 1: Setup and paths
1. Set the folder that holds your videos.
2. Choose how many frames you want per video.
3. Run the cell to install and import dependencies.

In [1]:
# Install dependencies
%pip -q install ultralytics opencv-python

# Imports
from pathlib import Path
import os
import random
import shutil
import time
import yaml
# import cv2
import numpy as np
from ultralytics import YOLO


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Config
#video_folder = Path("/path/to/videos")  # Folder containing your videos
#frames_per_video = 25  # X photos per video
#image_ext = ".jpg"
random_seed = 812
train_split = 0.8

# Output dataset folders
dataset_root = Path("/path/to/dataset")
images_train_dir = dataset_root / "images" / "train"
images_val_dir = dataset_root / "images" / "val"
labels_train_dir = dataset_root / "labels" / "train"
labels_val_dir = dataset_root / "labels" / "val"

for d in [images_train_dir, images_val_dir, labels_train_dir, labels_val_dir]:
   d.mkdir(parents=True, exist_ok=True)

## Step 2: Extract frames from videos
1. Put your videos in the folder you set above.
2. Run the next cell to extract frames into per-video subfolders.
3. Annotate those frames with your labeling tool (YOLO format).

In [ ]:
# def extract_frames_from_video(video_path: Path, frames_count: int, out_dir: Path, ext: str = ".jpg"):
#     cap = cv2.VideoCapture(str(video_path))
#     if not cap.isOpened():
#         print(f"Skipping {video_path.name}: cannot open")
#         return 0
#     total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
#     if total_frames <= 0:
#         print(f"Skipping {video_path.name}: no frames")
#         cap.release()
#         return 0
#     out_dir.mkdir(parents=True, exist_ok=True)
#     indices = np.linspace(0, total_frames - 1, frames_count, dtype=int)
#     saved = 0
#     for idx in indices:
#         cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
#         ok, frame = cap.read()
#         if not ok:
#             continue
#         out_file = out_dir / f"{video_path.stem}_f{idx:06d}{ext}"
#         cv2.imwrite(str(out_file), frame)
#         saved += 1
#     cap.release()
#     return saved

# video_files = sorted([p for p in video_folder.iterdir() if p.suffix.lower() in {".mp4", ".mov", ".avi", ".mkv"}])
# if not video_files:
#     print("No videos found. Check video_folder.")
# else:
#     for video_path in video_files:
#         out_dir = video_path.parent / f"{video_path.stem}_frames"
#         saved = extract_frames_from_video(video_path, frames_per_video, out_dir, image_ext)
#         print(f"{video_path.name}: saved {saved} frames to {out_dir}")

## Step 3: Prepare train/val folders
1. Collect your labeled frames and YOLO label files in one folder.
2. Set `labeled_root` to that folder and run the next cell.
3. You will get `images/train`, `images/val`, `labels/train`, `labels/val`.

In [3]:
# Folder that contains both images and YOLO .txt labels with matching names
labeled_root = Path("D:\\BR Footage\\Gharial\\target\\train_compiled")

# Class names in your dataset (order must match label IDs)
class_names = ["gharial"]

# If an image has no labels, create an empty .txt so it can be used as a negative example
allow_unlabeled = True

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

def list_image_label_pairs(root: Path, exts=IMAGE_EXTS, include_unlabeled: bool = True):
    pairs = []
    images_dir = root / "images"
    labels_dir = root / "labels"
    if images_dir.exists() and labels_dir.exists():
        label_map = {p.stem: p for p in labels_dir.rglob("*.txt")}
        for img_path in images_dir.rglob("*"):
            if img_path.suffix.lower() in exts:
                label_path = label_map.get(img_path.stem)
                if label_path and label_path.exists():
                    pairs.append((img_path, label_path))
                elif include_unlabeled:
                    pairs.append((img_path, None))
        return pairs
    for img_path in root.rglob("*"):
        if img_path.suffix.lower() in exts:
            label_path = img_path.with_suffix(".txt")
            if label_path.exists():
                pairs.append((img_path, label_path))
            elif include_unlabeled:
                pairs.append((img_path, None))
    return pairs

pairs = list_image_label_pairs(labeled_root, include_unlabeled=allow_unlabeled)
if not pairs:
    image_count = sum(1 for p in labeled_root.rglob("*") if p.suffix.lower() in IMAGE_EXTS)
    label_count = sum(1 for p in labeled_root.rglob("*.txt"))
    raise RuntimeError(
        f"No images found in {labeled_root}. "
        f"Found {image_count} images and {label_count} labels. "
        "Ensure the folder contains images (and optional .txt labels) or use images/ and labels/ subfolders."
    )

random.seed(random_seed)
random.shuffle(pairs)
split_idx = int(len(pairs) * train_split)
train_pairs = pairs[:split_idx]
val_pairs = pairs[split_idx:]

def copy_pairs(pairs_list, images_dir: Path, labels_dir: Path):
    labels_dir.mkdir(parents=True, exist_ok=True)
    for img_path, label_path in pairs_list:
        shutil.copy2(img_path, images_dir / img_path.name)
        out_label = labels_dir / f"{img_path.stem}.txt"
        if label_path is None:
            out_label.write_text("")
        else:
            shutil.copy2(label_path, out_label)

copy_pairs(train_pairs, images_train_dir, labels_train_dir)
copy_pairs(val_pairs, images_val_dir, labels_val_dir)

data_yaml = {
    "path": str(dataset_root),
    "train": "images/train",
    "val": "images/val",
    "nc": len(class_names),
    "names": class_names,
}
with open(dataset_root / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(f"Train images: {len(train_pairs)} | Val images: {len(val_pairs)}")
print(f"Wrote {dataset_root / 'data.yaml'}")

Train images: 998 | Val images: 250
Wrote \path\to\dataset\data.yaml


## Step 4: Train and validate
1. Choose a base model or let the notebook pick the most recent run.
2. Run training and validation on your new dataset.
3. Your best weights are saved under `runs/detect/train*/weights/`.

In [4]:
def find_latest_weights(search_dir: Path) -> Path | None:
    candidates = list(search_dir.rglob("*/weights/best.pt")) + list(search_dir.rglob("*/weights/last.pt"))
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)

runs_dir = Path("runs") / "detect"
latest_weights = find_latest_weights(runs_dir)
base_model = latest_weights if latest_weights else "yolov8s.pt"
print(f"Using base model: {base_model}")

model = YOLO(str(base_model))
train_results = model.train(
    data=str(dataset_root / "data.yaml"),
    epochs=15,
    imgsz=800,
    batch=16,
    project="runs/detect",
    name="train_custom",
    exist_ok=True,
)

val_results = model.val(data=str(dataset_root / "data.yaml"))
print(val_results)

Using base model: runs\detect\train_custom\weights\last.pt
New https://pypi.org/project/ultralytics/8.4.14 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.223  Python-3.11.9 torch-2.9.0+cpu CPU (AMD Ryzen AI 7 350 w/ Radeon 860M)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=\path\to\dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs\detect\train_custom\weights\last.p

In [ ]:
import matplotlib.pyplot as plt

# Extract training loss and epochs from the training results
losses = train_results.metrics['train/loss']
epochs = range(1, len(losses) + 1)

# Plot the training loss
plt.figure(figsize=(10, 6))
plt.plot(epochs, losses, marker='o', label='Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training Loss Over Epochs')
plt.legend()
plt.grid()
plt.show()

## Step 5: Get center pixel coordinates
1. Pick an image to test your detector.
2. Run inference with the trained weights.
3. The next cell prints $(x, y)$ centers for each detection.

In [ ]:
# Use your latest trained weights for inference
inference_weights = find_latest_weights(runs_dir) or base_model
infer_model = YOLO(str(inference_weights))

test_image = Path("D:\\BR Footage\\Gharial\\target\\validate\\01_05_2024 54")
results = infer_model.predict(source=str(test_image), imgsz=640, conf=0.25, verbose=False)

centers = []
for r in results:
    if r.boxes is None:
        continue
    for box in r.boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = box
        cx = (x1 + x2) / 2.0
        cy = (y1 + y2) / 2.0
        centers.append((float(cx), float(cy)))

print("Centers (x, y):")
for c in centers:
    print(c)

WARNING 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



error: OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\core\src\alloc.cpp:73: error: (-4:Insufficient memory) Failed to allocate 6220800 bytes in function 'cv::OutOfMemoryError'


## Step 6: Measure distance traveled
1. Set a folder of ordered frames and (optionally) an output CSV path.
2. Run detection across frames to compute center points and distances.
3. Review the printed table or saved CSV for totals and per-frame stats.

In [5]:
# Folder of frames in order (e.g., a *_frames folder or your val/train folder)
frames_folder = Path("D:\\BR Footage\\Gharial\\target\\validate\\master_frames")
output_csv = None  # Set to None to skip saving
target_class_id = 0
conf_threshold = 0.00
pixels_per_unit = None  # Set to e.g. 10.0 if you want distance in real-world units

# Ensure we have a model for inference
if "infer_model" not in globals():
    inference_weights = find_latest_weights(runs_dir) or base_model
    infer_model = YOLO(str(inference_weights))

def get_center_for_frame(img_path: Path):
    results = infer_model.predict(source=str(img_path), imgsz=640, conf=conf_threshold, verbose=False)
    best = None
    for r in results:
        if r.boxes is None or len(r.boxes) == 0:
            continue
        boxes = r.boxes
        cls = boxes.cls.cpu().numpy().astype(int)
        conf = boxes.conf.cpu().numpy()
        xyxy = boxes.xyxy.cpu().numpy()
        for i in range(len(cls)):
            if cls[i] != target_class_id:
                continue
            if best is None or conf[i] > best["conf"]:
                x1, y1, x2, y2 = xyxy[i]
                cx = (x1 + x2) / 2.0
                cy = (y1 + y2) / 2.0
                best = {"cx": float(cx), "cy": float(cy), "conf": float(conf[i])}
    return best

frame_paths = sorted([p for p in frames_folder.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
if not frame_paths:
    raise RuntimeError("No frames found. Check frames_folder.")

rows = []
prev = None
cum_dist = 0.0
for idx, img_path in enumerate(frame_paths):
    center = get_center_for_frame(img_path)
    if center is None:
        rows.append({
            "frame": img_path.name,
            "frame_index": idx,
            "center_x": None,
            "center_y": None,
            "confidence": None,
            "distance_from_prev": None,
            "cumulative_distance": cum_dist,
        })
        continue
    dist = None
    if prev is not None:
        dx = center["cx"] - prev["cx"]
        dy = center["cy"] - prev["cy"]
        dist = float((dx ** 2 + dy ** 2) ** 0.5)
        if pixels_per_unit:
            dist = dist / float(pixels_per_unit)
        cum_dist += dist
    rows.append({
        "frame": img_path.name,
        "frame_index": idx,
        "center_x": center["cx"],
        "center_y": center["cy"],
        "confidence": center["conf"],
        "distance_from_prev": dist,
        "cumulative_distance": cum_dist,
    })
    prev = center

# Display table (pandas if available, otherwise print rows)
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df)
    if output_csv:
        df.to_csv(output_csv, index=False)
        print(f"Saved CSV to {output_csv}")
except Exception:
    for row in rows[:5]:
        print(row)
    if output_csv:
        import csv
        with open(output_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)
        print(f"Saved CSV to {output_csv}")

,frame,frame_index,center_x,center_y,confidence,distance_from_prev,cumulative_distance
0,01050110_0.000.png,0,49.902660,726.199463,0.007852,NaN,0.000000e+00
1,01050110_0.000_04e59a6a-0add-4c90-825f-c5343de...,1,49.902660,726.199463,0.007852,0.000000,0.000000e+00
2,frame_0001.jpg,2,1753.467163,811.859741,0.031569,1705.716770,1.705717e+03
3,frame_0001_01dd8131-efcc-491a-a755-a12c5da8a8b...,3,219.768311,107.666611,0.000558,1687.637442,3.393354e+03
4,frame_0001_021410a9-4f49-4788-a74a-640f7160b9a...,4,428.960083,819.111328,0.328727,741.562394,4.134917e+03
...,...,...,...,...,...,...,...
1301,frame_0002_fe0c541d-e87f-4393-bcf8-32e8110cb6e...,1301,1711.934448,865.341614,0.000628,1665.675893,1.025593e+06
1302,frame_0002_fe2ce13d-fd7e-4df1-a393-9e782545dcb...,1302,1575.779785,848.991821,0.756266,137.132812,1.025730e+06
1303,frame_0002_ff65b8a3-f83c-4339-87d1-7afd44f1037...,1303,1580.717285,956.232910,0.020209,107.354693,1.025837e+06
1304,frame_0002_fff4e5ad-18d4-4756-9082-598061b42b6...,1304,647.372314,793.983032,0.626855,947.342524,1.026785e+06


In [6]:
df['pacing'] = df['distance_from_prev'].apply(lambda d: 1 if d is not None and d > 3 else 0)

In [7]:
df[df["center_x"].notnull()].sort_values("confidence", ascending=False)

,frame,frame_index,center_x,center_y,confidence,distance_from_prev,cumulative_distance,pacing
394,frame_0001_95e073ab-8929-4f06-8995-e1d12a38f6c...,394,629.829529,869.335876,0.846734,264.564696,301208.466981,1
675,frame_0002_089a1ebe-b207-436b-92b1-eb1bf45b215...,675,633.563782,866.608521,0.842240,594.719794,522943.760246,1
1077,frame_0002_ab27a6df-d294-414e-b4b6-c5f086c8a10...,1077,345.435608,900.640381,0.837746,131.522390,856588.495269,1
104,frame_0001_254c9716-93a0-4e0e-b987-f18216e958f...,104,341.265839,896.333130,0.836498,150.809839,77916.036737,1
282,frame_0001_6c59fa6c-d501-4190-8332-f75672b5789...,282,1586.203369,828.547485,0.834199,1507.665054,214226.602406,1
...,...,...,...,...,...,...,...,...
804,frame_0002_3df61b9c-9507-4491-8032-1b87ebb521f...,804,53.296761,1064.193237,0.000040,439.432102,630533.421939,1
984,frame_0002_8aeeefe2-f653-402c-8277-fd327e7ec35...,984,1733.834717,134.836731,0.000039,1452.820883,779594.062456,1
283,frame_0001_6cad9370-551a-47c5-89fc-c25579199e2...,283,1460.843018,739.094238,0.000037,154.003575,214380.605981,1
402,frame_0001_9a1be409-748a-420d-964c-b5202aeb420...,402,46.218723,76.292770,0.000036,1277.345457,309507.497502,1
